<a href="https://colab.research.google.com/github/AishwaryaVijay24/AishwaryaVijay24/blob/main/01_llm_behavior_lab_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ! means: run this as a shell/terminal command inside Colab, not normal Python code.
# pip means: Python package installer.
# install means: download and install packages.
# -q means: quiet mode, so Colab shows less installation output.

!pip install -q transformers accelerate torch sentencepiece pandas gradio

In [ ]:
# transformers:
# Hugging Face library used to load tokenizers and language models.

# accelerate:
# Helps run models efficiently on CPU/GPU.

# torch:
# PyTorch library. Most Hugging Face models run using PyTorch.

# sentencepiece:
# Tokenizer dependency used by many language models.

# pandas:
# Used to create tables/DataFrames for token and experiment results.

# gradio:
# Used later to create a small web UI for the project.

In [ ]:
# import means: bring a library into this notebook so we can use it.

import torch
# torch is PyTorch.
# We use it for tensors, GPU checks, and running the model.

import pandas as pd
# pandas helps us create tables.
# "as pd" means we can write pd instead of pandas.
# Example: pd.DataFrame(...)

from transformers import AutoTokenizer, AutoModelForCausalLM
# from transformers means: take specific tools from the Hugging Face Transformers library.
# AutoTokenizer loads the correct tokenizer for the model automatically.
# AutoModelForCausalLM loads a causal language model.
# CausalLM means the model predicts the next token from previous tokens.

In [ ]:
# MODEL_ID is a variable.
# It stores the Hugging Face model name as text/string.

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
# HuggingFaceTB is the organization/user name on Hugging Face.
# SmolLM2-135M-Instruct is the model name.
# 135M means around 135 million parameters.
# Instruct means the model was tuned to follow instructions better.

In [ ]:
# torch.cuda.is_available() checks whether Colab has a GPU available.
# If GPU is available, we use "cuda".
# Else, we use "cpu".

device = "cuda" if torch.cuda.is_available() else "cpu"

# print displays text/output in the notebook.
print("Using device:", device)

Using device: cuda


In [ ]:
# AutoTokenizer.from_pretrained(...) downloads/loads the tokenizer for MODEL_ID.
# tokenizer converts text into token IDs and token IDs back into text.

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

In [ ]:
# AutoModelForCausalLM.from_pretrained(...) downloads/loads the actual language model.
# MODEL_ID tells it which model to load.

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,

    # torch_dtype controls the number format used by the model.
    # float16 uses less GPU memory and is faster on GPU.
    # float32 is safer on CPU.
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
# .to(device) moves the model to GPU if device is cuda, otherwise CPU.

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
# Some models need a pad token for generation.
# pad_token is used when inputs need padding to equal length.
# eos_token means "end of sequence" token.

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# model.eval() puts the model in evaluation/inference mode.
# This means we are using it to generate answers, not train it.

model.eval()

print("Model loaded:", MODEL_ID)

Model loaded: HuggingFaceTB/SmolLM2-135M-Instruct


In [ ]:
# def means: define a function.
# inspect_tokens is the function name.
# text is the input parameter.
# The user will pass text into this function.

def inspect_tokens(text):

    # tokenizer.encode converts text into token IDs.
    # add_special_tokens=False means do not add model-specific extra tokens.
    # Example special tokens: beginning token, end token, role token, etc.

    token_ids = tokenizer.encode(text, add_special_tokens=False)

    # convert_ids_to_tokens converts token IDs into readable token pieces.
    # Tokens are not always full words.
    # Example: "playing" may become "play" + "ing".

    tokens = tokenizer.convert_ids_to_tokens(token_ids)

    # rows is an empty list.
    # We will store each token's information as a dictionary inside this list.

    rows = []

    # enumerate gives us index numbers.
    # zip pairs token_id and token together.
    # index = position of token in the text.
    # token_id = number used by the model.
    # token = readable token piece.

    for index, (token_id, token) in enumerate(zip(token_ids, tokens)):

        # append adds one new item into the rows list.
        # Each item is a dictionary with key-value pairs.

        rows.append({
            "index": index,
            "token_id": token_id,
            "token": token
        })

    # pd.DataFrame converts rows into a table.
    # return sends that table back as the function output.

    return pd.DataFrame(rows)

In [ ]:
# Here we call/run the function.
# The text inside quotes is passed into inspect_tokens.

inspect_tokens("LLMs do not know like a database. They predict the next token.")

,index,token_id,token
0,0,16421,LL
1,1,16653,Ms
2,2,536,Ġdo
3,3,441,Ġnot
4,4,699,Ġknow
5,5,702,Ġlike
6,6,253,Ġa
7,7,5817,Ġdatabase
8,8,30,.
9,9,1069,ĠThey


In [ ]:
# This function shows the most likely next tokens after a prompt.
# prompt = input text.
# top_k = how many top predictions we want to see.

def next_token_predictions(prompt, top_k=10):

    # tokenizer(...) converts prompt text into model input format.
    # return_tensors="pt" means return PyTorch tensors.
    # .to(device) moves input tensors to GPU/CPU.

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # torch.no_grad() means: do not calculate gradients.
    # Gradients are needed for training, but not for prediction.
    # This saves memory and makes inference faster.

    with torch.no_grad():

        # model(**inputs) sends the tokenized input into the model.
        # **inputs unpacks the input dictionary.
        # Example: input_ids=..., attention_mask=...

        outputs = model(**inputs)

    # outputs.logits contains raw prediction scores.
    # [0, -1, :] means:
    # 0 = first item in batch
    # -1 = last token position
    # : = all vocabulary tokens
    # So this gives scores for the next possible token.

    logits = outputs.logits[0, -1, :]

    # softmax converts raw logits into probabilities.
    # Probabilities are easier to understand.
    # Example: token A = 0.30, token B = 0.10

    probabilities = torch.softmax(logits, dim=-1)

    # torch.topk selects the top K highest probabilities.
    # top_probs = probability values.
    # top_ids = token IDs of those probabilities.

    top_probs, top_ids = torch.topk(probabilities, top_k)

    # Empty list to store table rows.

    rows = []

    # Loop through each token_id and its probability.

    for token_id, probability in zip(top_ids, top_probs):

        # token_id.item() converts PyTorch tensor value into normal Python number.

        token_number = token_id.item()

        # tokenizer.decode converts token ID back into readable text.

        token_text = tokenizer.decode([token_number])

        # probability.item() converts tensor probability into Python float.
        # round(..., 5) keeps only 5 decimal places.

        probability_value = round(probability.item(), 5)

        # Store one row of result.

        rows.append({
            "token_id": token_number,
            "token": token_text,
            "probability": probability_value
        })

    # Convert rows list into a table.

    return pd.DataFrame(rows)

In [ ]:
# This asks:
# After "The capital of France is", what tokens are most likely next?

next_token_predictions("The capital of France is", top_k=10)

,token_id,token,probability
0,7042,Paris,0.47192
1,260,the,0.26270
2,3807,located,0.08331
3,1217,called,0.01773
4,2250,Le,0.01494
5,253,a,0.00845
6,1343,known,0.00632
7,5145,La,0.00576
8,13010,situated,0.00549
9,3692,Mon,0.00504


In [ ]:
# This function generates text from a prompt.
# prompt = input text.
# temperature = controls randomness.
# top_p = controls token sampling pool.
# max_new_tokens = maximum number of tokens model can generate.

def generate_response(prompt, temperature=0.7, top_p=0.9, max_new_tokens=120):

    # Convert input prompt into tensors for the model.

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # no_grad because we are generating, not training.

    with torch.no_grad():

        # model.generate creates new tokens after the input prompt.

        output_ids = model.generate(
            **inputs,

            # max_new_tokens controls max generated output length.
            max_new_tokens=max_new_tokens,

            # do_sample=True means model samples from probabilities.
            # If False, model usually picks the most likely token every time.
            do_sample=True,

            # temperature controls randomness.
            # Low temperature = safer and more predictable.
            # High temperature = more creative but more unstable.
            temperature=temperature,

            # top_p means nucleus sampling.
            # Model chooses from the smallest set of tokens whose probabilities add up to top_p.
            # Example: top_p=0.9 means consider tokens covering 90% probability mass.
            top_p=top_p,

            # pad_token_id tells model what token to use for padding.
            pad_token_id=tokenizer.eos_token_id
        )

    # tokenizer.decode converts generated token IDs back to text.
    # output_ids[0] means first generated sequence.
    # skip_special_tokens=True removes special tokens from output.

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [ ]:
# Store prompt text in a variable.

prompt = "Explain next-token prediction in simple words."

In [ ]:
# Low temperature test.

print("LOW TEMPERATURE")
print(generate_response(prompt, temperature=0.1, top_p=0.9))

LOW TEMPERATURE
Explain next-token prediction in simple words.

The following code demonstrates the implementation of the `next_token` function:

```python
def next_token(word):
    if word in ["a", "b", "c"]:
        return word
    else:
        return "Unknown"
```

The `next_token` function takes a word as input and returns the next token in the sequence. If the word is not in the list of words, it returns "Unknown". Otherwise, it returns the word as a string.


In [ ]:
# Medium temperature test.

print("\nMEDIUM TEMPERATURE")
print(generate_response(prompt, temperature=0.7, top_p=0.9))


MEDIUM TEMPERATURE
Explain next-token prediction in simple words.

For example, if you have the following sentence: "The ball was thrown by John, and it landed in the park."

We can use the following model for prediction:

```python
model = Model(input_fn=lambda sentence: sentence.split(), output_fn=lambda output: output.split())
model.fit(input_fn, output_fn)
```

In this example, we define a model with input_fn and output_fn. We then use the `input_fn` function to split the input sentence into two


In [ ]:
# High temperature test.

print("\nHIGH TEMPERATURE")
print(generate_response(prompt, temperature=1.2, top_p=0.95))


HIGH TEMPERATURE
Explain next-token prediction in simple words.

How is one going to learn the 6,4,4 pattern before using an instance?


In [ ]:
# This function creates a chat-style interaction with system + user prompts.

def chat(system_prompt, user_prompt, temperature=0.7, top_p=0.9, max_new_tokens=150):

    # messages is a list of chat messages.
    # Each message has:
    # role    -> who is speaking
    # content -> what they are saying

    messages = [
        {
            "role": "system",        # system controls the assistant's behavior
            "content": system_prompt # system_prompt is the instruction we pass to the model
        },
        {
            "role": "user",          # user means actual user question
            "content": user_prompt   # user_prompt is the question/task
        }
    ]

    # apply_chat_template converts the messages into the exact chat format
    # expected by this model.
    #
    # tokenize=True means convert text into token IDs.
    # add_generation_prompt=True tells the model: now assistant should answer.
    # return_tensors="pt" means return PyTorch tensors.
    # return_dict=True means return a dictionary-like object containing input_ids etc.

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(device)

    # torch.no_grad() means we are not training.
    # We are only generating output, so gradients are not needed.

    with torch.no_grad():

        # model.generate creates new tokens.
        #
        # **inputs is important.
        # It unpacks the dictionary into:
        # input_ids=...
        # attention_mask=...
        #
        # This fixes your error.

        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id
        )

    # inputs["input_ids"] contains the original prompt tokens.
    # We only want the newly generated answer, not the whole prompt again.

    input_token_count = inputs["input_ids"].shape[-1]

    # output_ids[0] means first generated output.
    # [input_token_count:] removes the original prompt tokens.
    # So we keep only the assistant's newly generated answer.

    generated_tokens = output_ids[0][input_token_count:]

    # Decode token IDs back into readable text.
    # skip_special_tokens=True removes special model tokens.

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # return sends the response back to whoever called the function.

    return response

In [ ]:
# Call the chat function.

response = chat(
    system_prompt="You are a Java interview coach. Explain clearly with examples.",
    user_prompt="Explain polymorphism in Java with one simple example."
)

# Print the generated response.

print(response)

In Java, polymorphism is a feature that allows you to define methods that can be called with different types of objects. This is achieved through the use of overloaded methods.

Here's a simple example of polymorphism:

```java
// Define a method with different types of arguments
public class Person {
    private String name;
    private int age;

    public Person(String name, int age) {
        this.name = name;
        this.age = age;
    }

    public String getName() {
        return name;
    }

    public int getAge() {
        return age;
    }
}

// Define a method with the same parameters as the Person class
public void greet(Person person)


In [ ]:
# hallucination_questions is a list of questions.
# Some questions are impossible, private, future-based, or unknowable.
# We use them to test whether the model makes things up.

hallucination_questions = [
    "What was the exact revenue of Scorpio Group's internal Vessel Portal project in 2024?",
    "Who won the 2032 FIFA World Cup?",
    "Give me the exact private database schema of OpenAI's training system."
]

In [ ]:
# Loop through every question in hallucination_questions.

for question in hallucination_questions:

    # Print separator line.
    # "=" * 80 means repeat "=" 80 times.

    print("=" * 80)

    # Print current question.

    print("QUESTION:", question)

    # Ask the model to answer carefully.
    # This system prompt tries to reduce hallucination.

    answer = chat(
        system_prompt="Answer carefully. If the answer is not knowable, say you are not sure.",
        user_prompt=question,
        temperature=0.8,
        top_p=0.95
    )

    # Print model answer.

    print(answer)

QUESTION: What was the exact revenue of Scorpio Group's internal Vessel Portal project in 2024?
According to reports, the exact revenue of Scorpio Group's internal Vessel Portal project in 2024 was $1.9 million. However, this is an estimate and not known as exact, as there may be some discrepancies with the source and the estimate.
QUESTION: Who won the 2032 FIFA World Cup?
The 2032 FIFA World Cup was won by the Netherlands, led by Frank Lampard.
QUESTION: Give me the exact private database schema of OpenAI's training system.
Here is the exact private database schema of OpenAI's training system:

```python
# OpenAI OpenML library

# OpenAI's model specification
MODEL_SPECS = {
    'MNIST_DL': 'MNIST_DL',
    'CIFAR_10_2': 'CIFAR_10_2',
    'MNIST_DL': 'MNIST_DL',
    'CIFAR_10_200': 'CIFAR_10_200',
    'MNIST_DL_000000000': 'MNIST_DL_000000000',
    'MNIST


In [ ]:
# experiment_logs is an empty list.
# We will store each experiment result inside this list.

experiment_logs = []

In [ ]:
# run_experiment is a helper function.
# It runs chat(), stores the input settings and response, then returns the answer.

def run_experiment(system_prompt, user_prompt, temperature, top_p):

    # Generate response using our chat function.

    response = chat(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        temperature=temperature,
        top_p=top_p
    )

    # Save experiment details as a dictionary.
    # This makes it easy to compare outputs later.

    experiment_logs.append({
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
        "temperature": temperature,
        "top_p": top_p,
        "response": response
    })

    # Return the response so we can see it immediately.

    return response

In [ ]:
# Experiment 1:
# Low temperature and strict teacher style.

run_experiment(
    system_prompt="You are a strict technical teacher.",
    user_prompt="Explain context window in LLMs.",
    temperature=0.3,
    top_p=0.9
)

"In the context window of a machine learning model, it refers to the region of the input data that is not covered by the model's predictions. In other words, it is the area of the data where the model is not making accurate predictions.\n\nIn a typical machine learning model, the context window is the region of the data where the model is not making predictions. This can be achieved by using techniques such as:\n\n1. **Model selection**: Selecting a subset of the data that is not covered by the model's predictions.\n2. **Model training**: Training the model on the data that is not covered by the context window.\n3. **Contextual feature extraction**: Extracting the context information from the data that is not"

In [ ]:
# Experiment 2:
# Higher temperature and fun teacher style.

run_experiment(
    system_prompt="You are a fun Gen Z teacher.",
    user_prompt="Explain context window in LLMs.",
    temperature=0.9,
    top_p=0.95
)

'In the context window, it refers to a feature that allows the LLM to dynamically and continuously update the output for a particular input. This feature enables the LLM to keep track of the state of the model and adapt its parameters accordingly.\n\nThe context window is managed by the model architecture, which determines the dynamic behavior of the LLM. It provides the model with a way to update the parameters and keep track of the output, including the weights and biases, and the state of the model.\n\nThe context window can be configured using various methods, such as:\n\n1. **Initial state**: A default value of 0.\n\n2. **State update**: A value that is updated on every iteration of the model.\n'

In [ ]:
# Convert experiment_logs list into a pandas table.

df_logs = pd.DataFrame(experiment_logs)

# Show table in Colab.

df_logs

,system_prompt,user_prompt,temperature,top_p,response
0,You are a strict technical teacher.,Explain context window in LLMs.,0.3,0.90,In the context window of a machine learning mo...
1,You are a fun Gen Z teacher.,Explain context window in LLMs.,0.9,0.95,"In the context window, it refers to a feature ..."


In [ ]:
# Save experiment logs as a CSV file.
# index=False means do not save extra row numbers into the CSV.

df_logs.to_csv("llm_experiment_logs.csv", index=False)

In [ ]:
# Import gradio library.
# gradio helps create a simple web interface.

import gradio as gr

In [ ]:
# gradio_chat is a wrapper function.
# Gradio will call this function when user clicks submit.

def gradio_chat(system_prompt, user_prompt, temperature, top_p, max_new_tokens):

    # Reuse our existing chat function.

    return chat(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        temperature=temperature,
        top_p=top_p,
        max_new_tokens=max_new_tokens
    )

In [ ]:
# gr.Interface creates a simple app.
# fn means the function Gradio should run.
# inputs means UI input components.
# outputs means UI output component.
# title is app heading.
# description is app explanation.

demo = gr.Interface(
    fn=gradio_chat,
    inputs=[
        gr.Textbox(
            label="System Prompt",
            value="You are a helpful AI teacher."
        ),
        gr.Textbox(
            label="User Prompt",
            value="Explain temperature in LLMs."
        ),
        gr.Slider(
            0.1,
            1.5,
            value=0.7,
            step=0.1,
            label="Temperature"
        ),
        gr.Slider(
            0.1,
            1.0,
            value=0.9,
            step=0.05,
            label="Top-p"
        ),
        gr.Slider(
            20,
            300,
            value=120,
            step=10,
            label="Max New Tokens"
        )
    ],
    outputs=gr.Textbox(label="Assistant Response"),
    title="PromptScope: LLM Behavior Lab",
    description="Experiment with system prompts, user prompts, temperature, top-p, and LLM responses."
)

In [ ]:
# launch starts the Gradio app.
# share=True creates a temporary public link from Colab.

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bd7cd2c68d7191896d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
